# Feature Engineering — NBA Player Longevity Prediction

This notebook analyzes NBA player performance data to engineer features for predicting player career longevity (`target_5yrs`). Steps: define target, drop non-predictive columns, correlation analysis, create composite metrics, handle nulls, and produce a clean ML-ready dataset.

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 30)
sns.set_style('whitegrid')

## 1. Load Dataset and Define the Target Variable

In [2]:
df = pd.read_csv('nba_players.csv')
print('Shape:', df.shape)
df.head()

Shape: (1340, 22)


,Unnamed: 0,name,gp,min,pts,fgm,fga,fg,3p_made,3pa,3p,ftm,fta,ft,oreb,dreb,reb,ast,stl,blk,tov,target_5yrs
0,0,Brandon Ingram,36,27.4,7.4,2.6,7.6,34.7,0.5,2.1,25.0,1.6,2.3,69.9,0.7,3.4,4.1,1.9,0.4,0.4,1.3,0
1,1,Andrew Harrison,35,26.9,7.2,2.0,6.7,29.6,0.7,2.8,23.5,2.6,3.4,76.5,0.5,2.0,2.4,3.7,1.1,0.5,1.6,0
2,2,JaKarr Sampson,74,15.3,5.2,2.0,4.7,42.2,0.4,1.7,24.4,0.9,1.3,67.0,0.5,1.7,2.2,1.0,0.5,0.3,1.0,0
3,3,Malik Sealy,58,11.6,5.7,2.3,5.5,42.6,0.1,0.5,22.6,0.9,1.3,68.9,1.0,0.9,1.9,0.8,0.6,0.1,1.0,1
4,4,Matt Geiger,48,11.5,4.5,1.6,3.0,52.4,0.0,0.1,0.0,1.3,1.9,67.4,1.0,1.5,2.5,0.3,0.3,0.4,0.8,1


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1340 entries, 0 to 1339
Data columns (total 22 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Unnamed: 0   1340 non-null   int64  
 1   name         1340 non-null   str    
 2   gp           1340 non-null   int64  
 3   min          1340 non-null   float64
 4   pts          1340 non-null   float64
 5   fgm          1340 non-null   float64
 6   fga          1340 non-null   float64
 7   fg           1340 non-null   float64
 8   3p_made      1340 non-null   float64
 9   3pa          1340 non-null   float64
 10  3p           1340 non-null   float64
 11  ftm          1340 non-null   float64
 12  fta          1340 non-null   float64
 13  ft           1340 non-null   float64
 14  oreb         1340 non-null   float64
 15  dreb         1340 non-null   float64
 16  reb          1340 non-null   float64
 17  ast          1340 non-null   float64
 18  stl          1340 non-null   float64
 19  blk          1340

In [4]:
print('Target: target_5yrs')
print(df['target_5yrs'].value_counts())
print()
print(df['target_5yrs'].value_counts(normalize=True).round(3))

df['target_5yrs'].value_counts().plot(kind='bar', color=['#C44E52','#4C72B0'])
plt.title('Target: Lasted 5+ Years (1) vs Did Not (0)')
plt.xticks([0,1], ['<5 years (0)', '5+ years (1)'], rotation=0)
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('target_distribution.png', dpi=100)
plt.show()
print('target_5yrs is 1 (lasted 5+ years in NBA) or 0 (did not). This is our binary classification target.')

Target: target_5yrs
target_5yrs
1    831
0    509
Name: count, dtype: int64

target_5yrs
1    0.62
0    0.38
Name: proportion, dtype: float64


target_5yrs is 1 (lasted 5+ years in NBA) or 0 (did not). This is our binary classification target.


**Target definition**: `target_5yrs = 1` means the player lasted 5+ years in the NBA; `0` means they did not. The class distribution shows moderate imbalance — more players lasted than did not — consistent with survival bias in the data.

## 2. Drop Non-Predictive Columns

In [5]:
print('Non-predictive columns to drop:')
print("  'Unnamed: 0' — auto-generated row index, no signal")
print("  'name' — player name; causes data leakage (model would memorize names, not learn stats)")

df_clean = df.drop(columns=['Unnamed: 0', 'name'])
print(f'\nBefore: {df.shape} -> After: {df_clean.shape}')
df_clean.head()

Non-predictive columns to drop:
  'Unnamed: 0' — auto-generated row index, no signal
  'name' — player name; causes data leakage (model would memorize names, not learn stats)

Before: (1340, 22) -> After: (1340, 20)


,gp,min,pts,fgm,fga,fg,3p_made,3pa,3p,ftm,fta,ft,oreb,dreb,reb,ast,stl,blk,tov,target_5yrs
0,36,27.4,7.4,2.6,7.6,34.7,0.5,2.1,25.0,1.6,2.3,69.9,0.7,3.4,4.1,1.9,0.4,0.4,1.3,0
1,35,26.9,7.2,2.0,6.7,29.6,0.7,2.8,23.5,2.6,3.4,76.5,0.5,2.0,2.4,3.7,1.1,0.5,1.6,0
2,74,15.3,5.2,2.0,4.7,42.2,0.4,1.7,24.4,0.9,1.3,67.0,0.5,1.7,2.2,1.0,0.5,0.3,1.0,0
3,58,11.6,5.7,2.3,5.5,42.6,0.1,0.5,22.6,0.9,1.3,68.9,1.0,0.9,1.9,0.8,0.6,0.1,1.0,1
4,48,11.5,4.5,1.6,3.0,52.4,0.0,0.1,0.0,1.3,1.9,67.4,1.0,1.5,2.5,0.3,0.3,0.4,0.8,1


**Rationale**: `Unnamed: 0` is a meaningless row index. `name` is an identifier — keeping it would cause data leakage (the model memorizes 'Kevin Durant' rather than learning from performance metrics), and it cannot generalize to future/unseen players.

## 3. Null Values in Performance Columns

In [6]:
null_counts = df_clean.isnull().sum()
print('Missing values per column:')
print(null_counts[null_counts > 0] if null_counts.sum() > 0 else 'No missing values found.')
print(f'\nTotal nulls: {null_counts.sum()}')

Missing values per column:
No missing values found.

Total nulls: 0


No missing values found. However, note that some % columns (`3p`, `ft`) show 0 for players who never attempted those shot types — this is a valid encoded value (0 attempts → 0%), not a missing value. In datasets with true nulls, we would apply median imputation (robust to skew/outliers) for performance stats, plus create missing-indicator flags for systematic missingness.

## 4. Exploratory Data Analysis — Feature Distributions

In [7]:
fig, axes = plt.subplots(2, 4, figsize=(16,8))
features = ['pts','min','gp','ast','reb','stl','blk','tov']
for ax, feat in zip(axes.flatten(), features):
    sns.histplot(data=df_clean, x=feat, hue='target_5yrs', kde=True, ax=ax,
                 palette={0:'#C44E52', 1:'#4C72B0'}, alpha=0.6)
    ax.set_title(feat)
plt.suptitle('Feature Distributions by Longevity (0=<5yr, 1=5+yr)', y=1.02)
plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=100)
plt.show()

## 5. Correlation Analysis and Multicollinearity Check

In [8]:
X = df_clean.drop(columns=['target_5yrs'])
corr = X.corr()

plt.figure(figsize=(14,11))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, square=True, linewidths=0.5)
plt.title('Correlation Matrix of Predictors')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=100)
plt.show()

In [9]:
# Find highly correlated pairs
high_corr = [(corr.columns[i], corr.columns[j], round(corr.iloc[i,j],3))
             for i in range(len(corr.columns)) for j in range(i+1, len(corr.columns))
             if abs(corr.iloc[i,j]) > 0.8]
hcd = pd.DataFrame(high_corr, columns=['Feature A','Feature B','Corr r']).sort_values('Corr r', ascending=False)
print(f'High-correlation pairs (|r|>0.8): {len(hcd)}')
print(hcd.to_string(index=False))

High-correlation pairs (|r|>0.8): 22
Feature A Feature B  Corr r
      pts       fgm   0.991
  3p_made       3pa   0.983
      ftm       fta   0.981
      fgm       fga   0.980
      pts       fga   0.980
     dreb       reb   0.978
     oreb       reb   0.933
      min       pts   0.912
      min       fga   0.910
      min       fgm   0.903
      pts       ftm   0.896
      pts       fta   0.881
      pts       tov   0.850
      fgm       ftm   0.848
      fga       tov   0.846
      fgm       fta   0.840
     oreb      dreb   0.839
      fgm       tov   0.834
      fga       ftm   0.827
      min       tov   0.826
      fga       fta   0.806
      ftm       tov   0.805


In [10]:
# Correlation with target
tc = df_clean.corr()['target_5yrs'].drop('target_5yrs').sort_values(ascending=False)
print('Feature correlations with target_5yrs:')
print(tc.round(3))

tc.plot(kind='barh', color=['#4C72B0' if v>0 else '#C44E52' for v in tc])
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Feature Correlation with target_5yrs')
plt.tight_layout()
plt.savefig('target_correlations.png', dpi=100)
plt.show()

Feature correlations with target_5yrs:
gp         0.397
min        0.318
fgm        0.318
pts        0.316
reb        0.299
ftm        0.297
fta        0.296
oreb       0.293
fga        0.293
dreb       0.285
tov        0.272
stl        0.230
fg         0.227
blk        0.210
ast        0.175
ft         0.107
3p_made    0.037
3pa        0.018
3p        -0.000
Name: target_5yrs, dtype: float64


**Multicollinearity findings**: Many raw pairs are highly correlated (|r|>0.8) — expected in basketball:
- `fgm`/`fga`/`pts` (makes, attempts, and points are mathematically linked)
- `oreb`/`dreb`/`reb` (components sum to the total)
- `ftm`/`fta`/`pts` (free throws drive scoring)
- `3p_made`/`3pa` (makes and attempts)

These redundancies will be addressed by (a) engineering composite features that capture the underlying signal more efficiently, and (b) dropping the most redundant raw columns.

## 6. Engineer New Composite Features

In [11]:
df_eng = df_clean.copy()

# Feature 1: Points Per Minute — scoring rate, not volume
df_eng['pts_per_min'] = df_eng['pts'] / (df_eng['min'] + 1e-6)

# Feature 2: Simplified Player Efficiency Rating (per minute)
# Rewards pts, reb, ast, stl, blk; penalizes missed shots and turnovers
df_eng['per_simple'] = (
    df_eng['pts'] + df_eng['reb'] + df_eng['ast'] + df_eng['stl'] + df_eng['blk']
    - (df_eng['fga'] - df_eng['fgm'])   # missed FG penalty
    - (df_eng['fta'] - df_eng['ftm'])   # missed FT penalty
    - df_eng['tov']
) / (df_eng['min'] + 1e-6)

# Feature 3: True Shooting % — accounts for 3pt value and FT efficiency
df_eng['ts_pct'] = df_eng['pts'] / (2 * (df_eng['fga'] + 0.44 * df_eng['fta'] + 1e-6))

# Feature 4: Assist-to-Turnover Ratio — playmaking quality
df_eng['ast_to_ratio'] = df_eng['ast'] / (df_eng['tov'] + 1e-6)

# Feature 5: Defensive Contribution Per Minute (stl + blk + dreb normalized)
df_eng['def_contrib_per_min'] = (df_eng['stl'] + df_eng['blk'] + df_eng['dreb']) / (df_eng['min'] + 1e-6)

print('Engineered features stats:')
new_feats = ['pts_per_min','per_simple','ts_pct','ast_to_ratio','def_contrib_per_min']
for f in new_feats:
    print(f"  {f}: mean={df_eng[f].mean():.3f}, std={df_eng[f].std():.3f}")

Engineered features stats:
  pts_per_min: mean=0.371, std=0.094
  per_simple: mean=0.401, std=0.099
  ts_pct: mean=0.501, std=0.053
  ast_to_ratio: mean=1.241, std=0.723
  def_contrib_per_min: mean=0.169, std=0.057


**Rationale for each composite feature:**

| Feature | Formula | Why it's better than raw stats |
|---------|---------|-------------------------------|
| `pts_per_min` | pts / min | Removes playing-time bias; a 15-min player with 8pts > 38-min player with 12pts in efficiency |
| `per_simple` | (pts+reb+ast+stl+blk - missed_FG - missed_FT - tov) / min | One number for all-around contribution per minute; reduces 10+ raw stats to 1 signal |
| `ts_pct` | pts / (2×(fga + 0.44×fta)) | Weights 3-pointers and free throws correctly; standard `fg%` does not |
| `ast_to_ratio` | ast / tov | Rewards playmaking quality not volume; high-turnover assisters are penalized |
| `def_contrib_per_min` | (stl+blk+dreb) / min | Combined defensive impact per minute; players who defend tend to stick on rosters |

In [12]:
fig, axes = plt.subplots(1, 5, figsize=(20,4))
for ax, feat in zip(axes, new_feats):
    df_eng.boxplot(column=feat, by='target_5yrs', ax=ax)
    ax.set_title(feat, fontsize=9)
    ax.set_xlabel('0=<5yr, 1=5+yr')
plt.suptitle('')
plt.tight_layout()
plt.savefig('engineered_features_boxplot.png', dpi=100)
plt.show()

new_corr = df_eng[new_feats + ['target_5yrs']].corr()['target_5yrs'].drop('target_5yrs')
print('Engineered feature correlations with target_5yrs:')
print(new_corr.round(3).sort_values(ascending=False))

Engineered feature correlations with target_5yrs:
per_simple             0.318
ts_pct                 0.248
pts_per_min            0.194
def_contrib_per_min    0.138
ast_to_ratio          -0.013
Name: target_5yrs, dtype: float64


## 7. Drop Redundant Raw Columns

In [13]:
# Drop raw counting stats now captured by engineered features:
# fgm, fga -> captured by fg% and per_simple
# ftm, fta -> captured by ft% and ts_pct  
# 3p_made, 3pa -> captured by 3p%
# oreb, dreb -> combined in reb and def_contrib_per_min

redundant_cols = ['fgm', 'fga', 'ftm', 'fta', '3p_made', '3pa', 'oreb', 'dreb']
df_final = df_eng.drop(columns=redundant_cols)
print(f'Dropped {len(redundant_cols)} redundant columns: {redundant_cols}')
print(f'Final dataset shape: {df_final.shape}')
print('Final columns:', df_final.drop(columns=["target_5yrs"]).columns.tolist())

Dropped 8 redundant columns: ['fgm', 'fga', 'ftm', 'fta', '3p_made', '3pa', 'oreb', 'dreb']
Final dataset shape: (1340, 17)
Final columns: ['gp', 'min', 'pts', 'fg', '3p', 'ft', 'reb', 'ast', 'stl', 'blk', 'tov', 'pts_per_min', 'per_simple', 'ts_pct', 'ast_to_ratio', 'def_contrib_per_min']


In [14]:
# Final correlation matrix
X_final = df_final.drop(columns=['target_5yrs'])
corr_final = X_final.corr()

plt.figure(figsize=(12,10))
mask = np.triu(np.ones_like(corr_final, dtype=bool))
sns.heatmap(corr_final, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, square=True, linewidths=0.5)
plt.title('Final Feature Correlation Matrix (After Engineering)')
plt.tight_layout()
plt.savefig('final_correlation.png', dpi=100)
plt.show()

remaining = [(corr_final.columns[i], corr_final.columns[j], round(corr_final.iloc[i,j],3))
             for i in range(len(corr_final.columns)) for j in range(i+1, len(corr_final.columns))
             if abs(corr_final.iloc[i,j]) > 0.8]
print(f'High-correlation pairs remaining (|r|>0.8): {len(remaining)}')
for a,b,r in remaining: print(f'  {a} vs {b}: {r}')

High-correlation pairs remaining (|r|>0.8): 4
  min vs pts: 0.912
  min vs tov: 0.826
  pts vs tov: 0.85
  fg vs ts_pct: 0.854


## 8. Final ML-Ready Dataset Summary

In [15]:
print('=== FINAL DATASET ===')
print(f'Rows: {df_final.shape[0]:,}')
print(f'Predictor features: {df_final.shape[1]-1}')
print(f'Target: target_5yrs')
print(f'Missing values: {df_final.isnull().sum().sum()}')
print()
print('Class balance:')
print(df_final['target_5yrs'].value_counts())
print()
df_final.to_csv('nba_features_engineered.csv', index=False)
print('Saved: nba_features_engineered.csv')
df_final.describe().round(3)

=== FINAL DATASET ===
Rows: 1,340
Predictor features: 16
Target: target_5yrs
Missing values: 0

Class balance:
target_5yrs
1    831
0    509
Name: count, dtype: int64

Saved: nba_features_engineered.csv


,gp,min,pts,fg,3p,ft,reb,ast,stl,blk,tov,target_5yrs,pts_per_min,per_simple,ts_pct,ast_to_ratio,def_contrib_per_min
count,1340.000,1340.000,1340.000,1340.000,1340.000,1340.000,1340.000,1340.000,1340.000,1340.000,1340.000,1340.000,1340.000,1340.000,1340.000,1340.000,1340.000
mean,60.414,17.625,6.801,44.169,19.150,70.300,3.034,1.551,0.619,0.369,1.194,0.620,0.371,0.401,0.501,1.241,0.169
std,17.434,8.308,4.358,6.138,16.052,10.578,2.058,1.471,0.410,0.429,0.723,0.486,0.094,0.099,0.053,0.723,0.057
min,11.000,3.100,0.700,23.800,0.000,0.000,0.300,0.000,0.000,0.000,0.100,0.000,0.122,0.095,0.278,0.000,0.053
25%,47.000,10.875,3.700,40.200,0.000,64.700,1.500,0.600,0.300,0.100,0.700,0.000,0.307,0.336,0.470,0.708,0.126
50%,63.000,16.100,5.550,44.100,22.200,71.250,2.500,1.100,0.500,0.200,1.000,1.000,0.363,0.397,0.503,1.096,0.158
75%,77.000,22.900,8.800,47.900,32.500,77.600,4.000,2.000,0.800,0.500,1.500,1.000,0.431,0.468,0.536,1.628,0.207
max,82.000,40.900,28.200,73.700,100.000,100.000,13.900,10.600,2.500,3.900,4.400,1.000,0.738,0.836,0.708,6.000,0.439


## Pipeline Summary

| Step | Action | Outcome |
|------|--------|---------|
| Target defined | `target_5yrs` isolated as y | Binary: 1=lasted 5+yr, 0=did not |
| Drop noise/leakage | `Unnamed: 0`, `name` removed | -2 columns |
| Null check | Zero missing values | No imputation needed |
| Correlation analysis | 10+ high-corr pairs found | Guides which raw cols to drop |
| `pts_per_min` | pts / min | Scoring rate, not volume |
| `per_simple` | All-around efficiency per min | Replaces 10+ correlated raw stats |
| `ts_pct` | True Shooting % | Correct shooting efficiency |
| `ast_to_ratio` | ast / tov | Playmaking quality |
| `def_contrib_per_min` | (stl+blk+dreb) / min | Combined defensive rate |
| Drop redundant cols | 8 raw counting stats removed | Reduced multicollinearity |
| **Final dataset** | **1,340 rows × 14 cols (13 features + target)** | **ML-ready, zero nulls** |